In [ ]:
import scipy.stats as stats
import json
import os

In [19]:
scores_dir = 'qa_decode_run1'
scores = os.listdir(scores_dir)
scores.sort(key=lambda x: x[slice(x.find("_en") + 1, x.find("_en") + 5)], reverse=True)

In [20]:
scores

['gemma-3-4b-it_ratios_segment_enms_ntrex-128_results_scores.json',
 'gemma-3-4b-it_ratios_sequence_enms_ntrex-128_results_scores.json',
 'gemma-3-4b-it_none_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_ratios_segment_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_ratios_sequence_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_none_enms_ntrex-128_results_scores.json',
 'qwen3-4b_ratios_segment_enms_ntrex-128_results_scores.json',
 'qwen3-4b_ratios_sequence_enms_ntrex-128_results_scores.json',
 'qwen3-4b_none_enms_ntrex-128_results_scores.json']

In [29]:
_batches = ((1997) // 64)
batches = [slice(i * 64, (i+1) * 64) for i in range(_batches)]

In [31]:
(_batches) * 64

1984

In [5]:
models = {
    "gemma-3-4b-it": "Gemma-3-4B-IT",
    "qwen3-4b": "Qwen3-4B",
    "translategemma-4b-it": "TranslateGemma-4B-IT"
}

In [21]:
for i in range(0, len(scores), 3):
    scores[i:i+3] = scores[i:i+3][::-1]

scores

['gemma-3-4b-it_none_enms_ntrex-128_results_scores.json',
 'gemma-3-4b-it_ratios_sequence_enms_ntrex-128_results_scores.json',
 'gemma-3-4b-it_ratios_segment_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_none_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_ratios_sequence_enms_ntrex-128_results_scores.json',
 'translategemma-4b-it_ratios_segment_enms_ntrex-128_results_scores.json',
 'qwen3-4b_none_enms_ntrex-128_results_scores.json',
 'qwen3-4b_ratios_sequence_enms_ntrex-128_results_scores.json',
 'qwen3-4b_ratios_segment_enms_ntrex-128_results_scores.json']

In [6]:
code2name = {
    "enzh": "Chinese",
    "enfr": "French",
    "ende": "German",
    "enms": "Malay",
    "enko": "Korean"
}

In [11]:
metrics = ['COMET', 'XCOMET', 'CometKiwi', 'spBLEU', 'METEOR', 'Naturalness']

In [32]:
def results_table():
    ind = 0
    langs = set()
    print("\\toprule")
    for i, score_file in enumerate(scores):
        with open(f"{scores_dir}/{score_file}", "r") as file:
            results = json.load(file)

        teval_level = score_file.split('_')[1]
        if "ratio" in score_file:
            results = results['scaled']
        idx = score_file.index("_en") + 1 
        lang = slice(idx, idx+4)
        lang_pair = score_file[lang]
        model = score_file.split('_')[0]

        if ind == 0:
            head = f"Model\t&\t\\multicolumn{{3}}{{c|}}{{Semantic-based}}\t&\t\\multicolumn{{2}}{{c|}}{{N-gram-based}}\t&\t\\multicolumn{{2}}{{c|}}{{Naturalness}}\t&\t\\multirow{{2}}{{*}}{{Average}}\t\\\\"
            print(head)
            header = "\\quad + QE$_{{~\\text{{granularity}}}}$\t"
            for metric in metrics:
                if isinstance(results[metric], dict):
                    for met in results[metric]:
                        header += f"&\t{met.capitalize()}\t"
                elif not metric in head:
                    header += f"&\t{metric}\t"
                else:
                    header += "&\t"
            header += '&\t\\\\'
            print(header)
        if not lang_pair in langs:
            print('\\midrule')
            print(f"\\multicolumn{{10}}{{c}}{{{lang_pair}}}\t\\\\")
            print(f"\\midrule")
            langs.add(lang_pair)
        if teval_level == "none":
            row = f"{models[model]}\t"
        elif teval_level == "comet":
            row = f"\\quad + XCOMET\t"
        else:
            method = score_file.split('_')[2]
            row = f"\\quad + {teval_level}$_{{~\\text{{{method[:3]}}}}}$\t"
        
        total = 0
        for metric in metrics:
            if isinstance(results[metric], dict):
                for _, score in results[metric].items():
                    total += score
                    row += f"&\t{score:.2f}\t"
            else:
                score = results[metric]
                total += score
                row += f"&\t{score:.2f}\t"
        avg = total / len(metrics)
        row += f'&\t{avg:.2f}\t\\\\'

        print(row)
        ind += 1
    print("\\bottomrule")

In [33]:
results_table()

\toprule
Model	&	\multicolumn{3}{c|}{Semantic-based}	&	\multicolumn{2}{c|}{N-gram-based}	&	\multicolumn{2}{c|}{Naturalness}	&	\multirow{2}{*}{Average}	\\
\quad + QE$_{{~\text{{granularity}}}}$	&	COMET	&	XCOMET	&	CometKiwi	&	spBLEU	&	METEOR	&	Segment	&	Sequence	&	\\
\midrule
\multicolumn{10}{c}{enms}	\\
\midrule
Gemma-3-4B-IT	&	83.37	&	76.02	&	76.92	&	21.50	&	56.35	&	49.15	&	49.39	&	68.78	\\
\quad + ratios$_{~\text{seq}}$	&	83.83	&	77.20	&	78.69	&	21.50	&	55.90	&	49.09	&	49.12	&	69.22	\\
\quad + ratios$_{~\text{seg}}$	&	83.87	&	76.76	&	77.91	&	21.50	&	56.55	&	49.22	&	49.08	&	69.15	\\
TranslateGemma-4B-IT	&	85.90	&	82.54	&	80.99	&	21.59	&	52.49	&	49.55	&	49.86	&	70.49	\\
\quad + ratios$_{~\text{seq}}$	&	85.51	&	81.75	&	80.80	&	21.59	&	51.84	&	49.67	&	49.87	&	70.17	\\
\quad + ratios$_{~\text{seg}}$	&	85.11	&	80.69	&	79.95	&	20.78	&	52.46	&	49.65	&	49.80	&	69.74	\\
Qwen3-4B	&	79.32	&	67.53	&	72.48	&	15.91	&	46.49	&	49.15	&	48.01	&	63.15	\\
\quad + ratios$_{~\text{seq}}$	&	80.67	&	70.63	&	7